### Fractional Factorial

When Full Factorial is not possible

#### Toy Example

4 factors (A, B, C, D), D as interaction for design

In [ ]:
# Dependencies
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Settings
np.set_printoptions(precision=3)

In [ ]:
""" Generate Fractional Factorial Design """
# Define number of factors (excluding interaction)
n_factors = 4
levels = np.array([-1, 1])  # 2 levels, extreme values

# Step 1: Generate all 2^(k-1) combinations of X_1 to X_(k-1)
n_runs = ...  # 2^(k-1)
ABC = 2 * np.array([list(np.binary_repr(i, width=(...))) for i in range(n_runs)], dtype=int) - 1
design = pd.DataFrame(ABC, columns=['A', 'B', 'C'])  # Create pandas DataFrame

# Step 2: Define X_k as if an interaction (product)
design['D'] = ...  # Interaction

design.insert(0, 'Run', np.arange(1, n_runs + 1))
print(f"Fractional Factorial Design (2^({n_factors}-1), D=AxB)")
display(design)

In [ ]:
""" Simulate Toy Model """
np.random.seed(42)
noise = np.random.normal(0, 2, n_runs)

design['Y'] = (2
               + 2 * design['A']
               + 3 * design['B']
               - 2.5 * design['C']
               + 1.5 * design['D']
               + 0.5 * design['A'] * design['B']
               + noise)

display(design)

In [ ]:
from statsmodels.formula.api import ols

""" Fit model """
model = ols(formula='Y ~ A + B + C + D', data=design).fit()

""" Coefficients and Residuals """
print('Regression Coefficients:')
display(model.params)
print('Residual Summary:')
display(model.resid.describe())

fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot()
ax.scatter(model.fittedvalues, model.resid)
ax.axhline(0, color='r', linestyle='--', linewidth=1)
ax.set_xlabel('Fitted values')
ax.set_ylabel('Residuals')
_ = ax.set_title('Residual Plot')

In [ ]:
""" ANOVA Table """
import statsmodels.api as sm
for i in range(3):
    anova_table = sm.stats.anova_lm(model, typ=(i+1))

    print(f"ANOVA Table Type {'I'*(i+1)}")
    display(anova_table)

In [ ]:
from statsmodels.graphics.factorplots import interaction_plot

""" Plot main effects and interaction """
# Main effects
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(1, 2, 1)
ax.set_title("Main Effects on Y")
columns = ['A', 'B', 'C', 'D']
for col in columns:
    means = design.groupby(col)['Y'].mean()
    ax.plot(means.index, means.values, '-o', label=col)
ax.set_xlabel('Factor Level')
ax.set_ylabel('Mean Response')
ax.set_xticks([-1,0,1])
ax.legend()

# Interactions
ax = fig.add_subplot(1, 2, 2)
interaction_plot(design['A'], design['B'], design['Y'], colors=['red', 'blue'], ax=ax)
ax.set_title('Interaction: AxB')
ax.set_xticks([-1,0,1])
fig.tight_layout()

In [ ]:
""" Optimising from Obtained Model """

from scipy.optimize import minimize

def model_prediction(x):
    df = pd.DataFrame([x], columns=columns)
    return model.predict(df)[0]

# Initial guess (centre point)
x0 = np.zeros(n_factors).ravel()

# Bounds for each factor
bounds = [(-1, 1)] * n_factors

# Minimise the predicted response
res_min = minimize(model_prediction, x0=x0, bounds=bounds)
print('Minimum predicted Y at:', res_min.x)
print('Predicted Y:', res_min.fun)

# Maximise (by minimizing the negative)
res_max = minimize(lambda x: -model_prediction(x), x0=x0, bounds=bounds)
print('Maximum predicted Y at:', res_max.x)
print('Predicted Y:', -res_max.fun)